<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/banking77-intent-classification/notebooks/pretrained_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/TheFaridKarami/farid-ai-lab.git

In [65]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)

SEED = 42
set_seed(SEED)

In [66]:
PROJECT_ROOT = (Path("/content/farid-ai-lab")
 / "projects/nlp/banking77-intent-classification")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [67]:
print(PROJECT_ROOT.exists())
print(DATA_DIR.exists())
print(TRAIN_PATH.exists())
print(TEST_PATH.exists())

True
True
True
True


In [68]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Shapes:", train_df.shape, test_df.shape)
print(train_df.dtypes)
print("Nulls:\n", train_df.isnull().sum())
print(train_df.head())

counts = train_df["label"].value_counts()
print("Classes train/test:", train_df["label"].nunique(), test_df["label"].nunique())
print(counts.describe())
print(counts.tail(5))
print("Missing from test:", set(train_df["label"]) - set(test_df["label"]))

print("Duplicate rows:", train_df.duplicated().sum())
print("Duplicate texts train/test:", train_df["text"].duplicated().sum(), test_df["text"].duplicated().sum())
print("Train/test text overlap:", len(set(train_df["text"]) & set(test_df["text"])))
print("Texts with conflicting labels:", (train_df.groupby("text")["label"].nunique() > 1).sum())

Shapes: (10003, 2) (3080, 2)
text     object
label    object
dtype: object
Nulls:
 text     0
label    0
dtype: int64
                                                text         label
0                     I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived ...  card_arrival
2  I have been waiting over a week. Is the card s...  card_arrival
3  Can I track my card while it is in the process...  card_arrival
4  How do I know if I will get my card, or if it ...  card_arrival
Classes train/test: 77 77
count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: count, dtype: float64
label
lost_or_stolen_card         82
card_swallowed              61
card_acceptance             59
virtual_card_not_working    41
contactless_not_working     35
Name: count, dtype: int64
Missing from test: set()
Duplicate rows: 0
Duplicate texts train/test: 0 0
Trai

In [ ]:
train_df = pd.read_csv(TRAIN_PATH).drop_duplicates(subset="text").reset_index(drop=True)
test_df  = pd.read_csv(TEST_PATH)
assert len(train_df) == 9999, len(train_df)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    train_df["text"], train_df["label"],
    test_size=0.2, random_state=SEED, stratify=train_df["label"])

le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_val   = le.transform(y_val_raw)
y_test  = le.transform(test_df["label"])

NUM_LABELS = len(le.classes_)
label2id = {str(l): int(i) for i, l in enumerate(le.classes_)}
id2label = {i: l for l, i in label2id.items()}

train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": y_train.tolist()})
val_ds   = Dataset.from_dict({"text": X_val.tolist(),   "labels": y_val.tolist()})
test_ds  = Dataset.from_dict({"text": test_df["text"].tolist(), "labels": y_test.tolist()})
print(NUM_LABELS, len(train_ds), len(val_ds), len(test_ds))

In [61]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


MODEL_NAME = "distilbert/distilbert-base-uncased"

MAX_LENGTH = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_lengths = [len(ids) for ids in train_tokens["input_ids"]]

train_tokens  = tokenizer(X_train.tolist(), add_special_tokens=True, truncation = False).to(DEVICE)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels= 77, label2id = label2idx, id2label=idx2label).to(DEVICE)



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [62]:
from sklearn.metrics import accuracy_score, f1_score


def compute_evaluation(eval_pred):
  logits, labels = eval_pred

  predictions = np.argmax(logits, axis=-1)

  accuracy = accuracy_score(labels, predictions)
  f1 = f1_score(labels, predictions)

  return {
      "Accuracy": accuracy,
      "F1 Score": f1
  }

## Mean: 16.28
## Median :  13.0
## Quantile result:
## 25% => 11, 50% => 13, 75% =>17, 90% =>28, 95% =>37, 99% 54, 100% =>93

So I decided to choose MAX_LEGNTH=64





In [64]:
def tokenize_function(dsataset):
  return tokenizer(
      dataset,
      truncation =True,
      return_tensor = "pt",
      add_special_tokens = False,
      max_length= MAX_LENGTH
  )

In [ ]:
train_dataset =